# LLM summary
This notebook showcases a simple engineered prompt used to summarize a long piece of text given in a pdf format.

In [54]:
from langchain_community.chat_models import ChatOllama
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

import pytesseract
from pdf2image import pdfinfo_from_path, convert_from_path
from tqdm import tqdm

Make sure to run an ollama server locally somewhere prior to this step via `ollama serve`. Alternatively, you may use GPT given an API key being stored as a local variable (pro tip: use .env for convenience). GPT is typically faster.

In [51]:
# llm = ChatOllama(model="llama3.2", temperature=0)
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

We then load in the entire PDF, which is likely to be well over the LLM's context limit. The code below assumes each pdf's page to be static images, using OCR to extract textual information.

In [11]:
pdf_file = 'test-documents/the-vampyre.pdf'
info = pdfinfo_from_path(pdf_file, userpw=None, poppler_path=None)

maxPages = info["Pages"]
page_data = []
for page in tqdm(range(1, maxPages+1)):
    page_img = convert_from_path(pdf_file, dpi=200, first_page=page, last_page = page)[0]
    page_text = pytesseract.image_to_string(page_img)
    page_data.append(page_text)

  0%|          | 0/25 [00:00<?, ?it/s]

100%|██████████| 25/25 [04:47<00:00, 11.51s/it]


Option 1: summarize each page (or group of pages not exceeding the token limit), then concatenate any future story to the summary

In [65]:
summary = ""
summary_format = '''
Characters:
    - (name) (description) (personality)
    (only list main characters)
Plot Overview:
    Summarize, in short, the story up to the current point in the story so that subsequent reader may pick up from here.
    Must be in one concise paragraph, not bullet points. Max 5 sentences.
Key events:
    - list key events up to this point as bullet points (i.e. A meeting B, C dying)
    make sure to mention the person (who), action (what), and place (where) if said information can be found
    (at most 20 events, concatenate existing ones once more story is added)
Observations:
    - list interesting traits of each characters, as well as events that leads to such an observation. 
    (at most 20 events, concatenate existing ones once more story is added)
Conclusion:
    - if the story were to end here, what event would you consider the ending to be (i.e. the main villain dies? the main protagonist wins?)?
'''
with open('prompts/story_summary.txt', 'r') as file:
    prompt_txt = file.read()

In [66]:
### Generate

# Prompt
prompt = PromptTemplate(
    template=prompt_txt,
    input_variables=["summary", "summary_format", "story"],
)

# Chain
rag_chain = prompt | llm | StrOutputParser()

# Chunk and Generate
n = 5
for i in tqdm(range(0, len(page_data), n)):
    story_chunk = "".join(page_data[i:min(i+n, len(page_data))])
    summary = rag_chain.invoke({
        "summary": summary,
        "summary_format": summary_format,
        "story": story_chunk
    })

100%|██████████| 5/5 [00:40<00:00,  8.01s/it]


In [67]:
print(summary)

Characters:
    - Lord Ruthven: A nobleman with a deadly hue and a chilling presence, he is enigmatic and feared, often leaving others in awe. His actions suggest a dark nature, as he shows a preference for the immoral and the corrupt.
    - Aubrey: A young, wealthy gentleman with a romanticized view of life, he is naive and idealistic, believing in the inherent goodness of people. His curiosity about Lord Ruthven leads him into a dangerous friendship.
    - Ianthe: A beautiful and innocent Greek girl, she captivates Aubrey with her charm and purity, contrasting sharply with the corrupt world he has encountered.
    - Miss Aubrey: Aubrey's sister, who possesses a melancholy charm and a deep connection with her brother, providing him solace amidst his turmoil.

Plot Overview:
    After separating from Lord Ruthven, Aubrey travels through Greece, where he becomes enchanted by Ianthe, a local girl. However, tragedy strikes when Aubrey witnesses the brutal death of a woman, believed to be 